# LexChain LLM legal-analysis benchmark — model selection under ZDR

**Research question:** which model is best at legal analysis. **Single variable = the model.** Cross-family instruct models accessed through the **same OpenRouter API** under a **zero-data-retention** policy (API-only, no self-hosting).

**Selecting the third model (this pass):** `mixtral-8x22b` was dropped (no ZDR provider on OpenRouter). Two distinct-family candidates are on trial — keep whichever routes cleanly under ZDR:

| Short name | Model | Family | Tier note |
|---|---|---|---|
| `llama-3.3-70b` | `meta-llama/llama-3.3-70b-instruct` | Meta | 70B dense (kept) |
| `qwen-2.5-72b` | `qwen/qwen-2.5-72b-instruct` | Qwen | 72B dense (kept) |
| `gemma3-27b` | `google/gemma-3-27b-it` | Google | 27B (below tier) — **trial** |
| `deepseek-v3.1` | `deepseek/deepseek-chat-v3.1` | DeepSeek | MoE ~37B active (tier mismatch) — **trial** |

**Privacy:** every request sends `provider = {data_collection:"deny", allow_fallbacks:false, zdr:true}`; a model with no ZDR provider returns 404 (fail loud). Turn prompt logging OFF in your OpenRouter account too.

Everything else frozen: prompt v2.0-cuad-checklist, temperature 0, JSON schema/parser, 10 seed-42 docs, reusable `ground_truth_key.csv`, blind build + SummEval + entity/risk F1 + aggregation. **No GPU needed.** Set Colab secret `OPENROUTER_API_KEY`.

In [ ]:
# Cell 1 — mount Drive, clone repo, install deps, load OpenRouter key
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
if not os.path.exists('/content/lexchain-chunk-embed-bench'):
    !git clone https://github.com/MarvinPescos/lexchain-chunk-embed-bench.git /content/lexchain-chunk-embed-bench
%cd /content/lexchain-chunk-embed-bench
!git pull
!pip install -q -r requirements-analysis.txt
os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
print('OpenRouter key loaded:', bool(os.environ.get('OPENROUTER_API_KEY')))

In [ ]:
# Cell 2 — remove stale checkpoints from the previous (self-hosted) model set,
# so no de-registered model can leak into the blind sheet or results
import glob, json, os
from analysis.data import MODELS
base = '/content/drive/MyDrive/lexchain_bench/analysis'
removed = 0
for f in glob.glob(f'{base}/analyses/*.json'):
    if json.load(open(f))['model'] not in MODELS:
        os.remove(f); removed += 1
for f in glob.glob(f'{base}/raw_failures/*'):
    os.remove(f)
print(f'removed {removed} stale checkpoints; current models: {list(MODELS)}')

In [ ]:
# Cell 3 — data + ground-truth authoring materials + CONTEXT SAFETY CHECK
# (no model output is produced or shown here)
!python download_data.py
!python -m analysis.prepare_ground_truth

import sys; sys.path.insert(0, '.')
from analysis.data import load_docs, sample_stems, context_check
docs = load_docs(); stems = sample_stems(docs.keys(), n=10)
report = context_check({s: docs[s] for s in stems})  # raises loudly on any misfit
print('CONTEXT CHECK PASSED — all 10 docs fit all models.')
for m, info in report['per_model'].items():
    print(f"  {m}: worst case {info['worst_case_tokens']:,} / limit {info['limit']:,}")

## ✋ GT-AUTHORING GATE — reuse your existing key

If you already authored `ground_truth_key.csv`, it is **unchanged and reusable** (same 10 docs). Confirm it is in `MyDrive/lexchain_bench/analysis/`. If not, author it now from `doc_texts/*.txt` (blind to model outputs) and save as `ground_truth_key.csv`.

Cells 4–5 only produce checkpoints (status/latency logs, no content). Cell 6 — the first cell that *renders* outputs — refuses to run until the key exists.

In [ ]:
# Cell 4 — ZDR selection: probe all 4 trial models, then a tolerant 2-doc smoke
# so a ZDR-404 model is REPORTED per-model instead of aborting the run.
!python -m analysis.probe_zdr --models \
meta-llama/llama-3.3-70b-instruct,qwen/qwen-2.5-72b-instruct,\
google/gemma-3-27b-it,deepseek/deepseek-chat-v3.1

# 2 docs x 4 models; --tolerate-failures records per-model pass/fail (see summary)
!python -m analysis.analyze --smoke 2 --tolerate-failures

import json
from pathlib import Path
from analysis.data import MODELS
cache = Path('/content/drive/MyDrive/lexchain_bench/analysis')
print("\n=== per-model ZDR smoke result ===")
for m in MODELS:
    recs = [json.loads(p.read_text()) for p in (cache/'analyses').glob(f'*__{m}.json')]
    ok = sum(r.get('ok') for r in recs)
    failed = [r for r in recs if r.get('call_failed')]
    if failed:
        print(f"  {m:16s} FAILED  ({failed[0].get('error','')[:80]})")
    else:
        lat = [r.get('latency_s') or 0 for r in recs]
        print(f"  {m:16s} {ok}/{len(recs)} ok  ~{sum(lat)/max(len(lat),1):.1f}s/doc")
print("\nKeep Llama + Qwen + whichever of gemma3-27b / deepseek-v3.1 routed cleanly")
print("(if both: prefer deepseek-v3.1 for max family distance; if neither: ship 2).")

In [ ]:
# Cell 5 — full run: 10 docs x 3 models = 30 outputs (resumable; reruns skip)
!python -m analysis.analyze

In [ ]:
# Cell 6 — blind sheet (first output-rendering step; guardrail-enforced)
!python -m analysis.build_blind_eval
print('\nblind_eval.csv: 30 rows, Output A–C labels, SummEval columns')
print('Rate coherence / consistency / fluency / relevance (1–5) + hallucination notes.')
print('Do NOT open unblinding_key.csv until all rating is done.')

### Human step (off-Colab)
Rate all 30 rows of `blind_eval.csv` on the **SummEval** dimensions: coherence, consistency, fluency, relevance (1–5). Then run Cell 7.

In [ ]:
# Cell 7 — aggregate: un-blind, entity/risk F1 vs your key, wins, final table
!python -m analysis.aggregate_analysis